# # S-DoT Paper Reproduction02

cell1 약 2분 소요

In [1]:
import pandas as pd
import numpy as np

merge_key = '시리얼_matched'  # 01번 노트북에서 썼던 키 이름 그대로

merged_df = pd.read_csv('../data/processed_merged_df.csv', low_memory=False)
sensor_coords = pd.read_csv('../data/processed_sensor_coords.csv')

print("merged_df:", merged_df.shape)
print("sensor_coords:", sensor_coords.shape)
sensor_coords.head()

merged_df: (4459253, 65)
sensor_coords: (1154, 3)


,시리얼_matched,위도,경도
0,OC3CL200304,37.495775,126.954450
1,OC3CL200305,37.504948,126.938966
2,OC3CL200012,37.544002,127.069731
3,OC3CL200013,37.583469,126.982622
4,OC3CL200014,37.563574,126.984504


#  Haversine 벡터화 계산

In [2]:
def haversine_vectorized(lat1, lon1, lat2, lon2):
    R = 6371000
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

lats = sensor_coords['위도'].values
lons = sensor_coords['경도'].values
n = len(sensor_coords)

lat_matrix1 = lats.reshape(-1, 1)
lat_matrix2 = lats.reshape(1, -1)
lon_matrix1 = lons.reshape(-1, 1)
lon_matrix2 = lons.reshape(1, -1)

dist_matrix = haversine_vectorized(lat_matrix1, lon_matrix1, lat_matrix2, lon_matrix2)
print("거리 행렬 shape:", dist_matrix.shape)

거리 행렬 shape: (1154, 1154)


# 500m 이내 directional pair 추출 + 저장

In [3]:
within_range = np.argwhere((dist_matrix <= 500) & (dist_matrix > 0))

pairs = []
for i, j in within_range:
    pairs.append((sensor_coords.iloc[i][merge_key], sensor_coords.iloc[j][merge_key]))

print(f"500m 이내 방향성 센서쌍: {len(pairs)}개")

pd.DataFrame(pairs, columns=['sensor_i', 'sensor_j']).to_csv(
    '../data/sensor_pairs_500m.csv', index=False, encoding='utf-8-sig'
)

500m 이내 방향성 센서쌍: 2806개


# 전처리
- 시간별 관측 데이터는 요소별 품질 마스크를 적용하여 (635, 488, 24) 형태의 센서 × 일 × 시간 텐서로 구성

In [4]:
# 1. 온도 컬럼 숫자형 변환 
merged_df['측정시간'] = pd.to_datetime(merged_df['측정시간'], errors='coerce')
merged_df['온도 평균(℃)'] = pd.to_numeric(merged_df['온도 평균(℃)'], errors='coerce')

# 2. 물리적으로 불가능한 값(이상치) 마스킹 - 온도 기준 예시
#    S-DoT 여름철 온도라면 대략 -10~50도 범위를 벗어나면 센서 오류로 간주
valid_range = (merged_df['온도 평균(℃)'] >= -10) & (merged_df['온도 평균(℃)'] <= 50)
merged_df.loc[~valid_range, '온도 평균(℃)'] = np.nan

print(f"이상치로 마스킹된 행: {(~valid_range).sum()}개")

# 3. 센서 x 시간 pivot (기존과 동일)
temp_pivot = merged_df.pivot_table(
    index='측정시간', columns=merge_key, values='온도 평균(℃)', aggfunc='mean'
)

# 4. 핵심: 시간 격자를 고정 (측정 주기에 맞춰 완전한 시간 인덱스 생성)
#    측정 주기 확인 먼저
time_diffs = temp_pivot.index.to_series().diff().dropna()
print("측정 주기(최빈값):", time_diffs.mode()[0])

이상치로 마스킹된 행: 546013개
측정 주기(최빈값): 0 days 01:00:00


In [5]:
original_na = merged_df['온도 평균(℃)'].isna().sum()
print(f"원래 결측치: {original_na}개")

원래 결측치: 546013개


- 온도 546,013개(12.2%)는 그냥 원래부터 없던 값(결측)   
- 센서가 그 시간에 아예 측정 X

In [6]:
# 1. 날짜와 시간 컬럼 분리
merged_df['날짜'] = merged_df['측정시간'].dt.date
merged_df['시간'] = merged_df['측정시간'].dt.hour

# 2. 주간 데이터 (day_df) 정의 (10:00 ~ 17:00)
day_mask = (merged_df['시간'] >= 10) & (merged_df['시간'] <= 17)
day_df = merged_df[day_mask]

# 3. 야간 데이터 (night_df) 정의 (22:00 ~ 06:00)
night_mask = (merged_df['시간'] >= 22) | (merged_df['시간'] <= 6)
night_df = merged_df[night_mask].copy()

# 4. 야간 데이터 날짜 보정 (00시~06시 데이터는 전날 야간으로 취급)
night_df['야간기준날짜'] = night_df['측정시간'].dt.date
is_past_midnight = night_df['시간'] <= 6
night_df.loc[is_past_midnight, '야간기준날짜'] = (night_df['측정시간'] - pd.Timedelta(days=1)).dt.date

print(f"주간 데이터(day_df) 생성 완료: {day_df.shape}")
print(f"야간 데이터(night_df) 생성 완료: {night_df.shape}")

주간 데이터(day_df) 생성 완료: (1300407, 67)
야간 데이터(night_df) 생성 완료: (1905414, 68)


# HW x day 계산

In [7]:
# 1. '날짜'로만 그룹화하여 10~17시 사이의 해당 일자 대푯값 산출
# 논문의 '일 최고 기온' 의도에 따라 모든 센서의 평균을 내거나 최댓값을 구함
day_temp = day_df.groupby('날짜')['온도 평균(℃)'].mean().reset_index() 

# 2. 당일 온도가 33도 이상인지 1차 판별
day_temp['is_over_33'] = day_temp['온도 평균(℃)'] >= 33

# 3. d일과 d-1일 모두 33도 이상인 조건 
# 오늘(is_over_33)이 True이고, 어제(shift(1))도 True일 경우에만 HW_day로 판정
day_temp['is_HW_day'] = day_temp['is_over_33'] & day_temp['is_over_33'].shift(1, fill_value=False)

# 4. ND x day 분류 (여집합)
day_temp['is_ND_day'] = ~day_temp['is_HW_day']

print("--- 주간(Day) 분류 결과 ---")
print(day_temp['is_HW_day'].value_counts())

--- 주간(Day) 분류 결과 ---
is_HW_day
False    197
True      12
Name: count, dtype: int64


# HW x night 계산

In [8]:
# 1. '야간기준날짜'로만 그룹화하여 해당 일자 야간 대푯값 산출
night_temp = night_df.groupby('야간기준날짜')['온도 평균(℃)'].mean().reset_index()

# 2. 야간 25도 도달 여부 확인
night_temp['is_HW_night'] = night_temp['온도 평균(℃)'] >= 25

# 3. ND x night 분류 (여집합)
night_temp['is_ND_night'] = ~night_temp['is_HW_night']

print("\n--- 야간(Night) 분류 결과 ---")
print(night_temp['is_HW_night'].value_counts())


--- 야간(Night) 분류 결과 ---
is_HW_night
True     125
False     86
Name: count, dtype: int64


# ND x day & ND x night

In [12]:
# 1. 주간(day) ND 분류: HW_day가 아닌(False) 날들을 True로 변환
day_temp['is_ND_day'] = ~day_temp['is_HW_day']

# 2. 야간(night) ND 분류: HW_night가 아닌(False) 날들을 True로 변환
night_temp['is_ND_night'] = ~night_temp['is_HW_night']

print("\n--- 최종 ND 분류 결과 ---")
print("주간 ND 일수:", day_temp['is_ND_day'].sum())
print("야간 ND 일수:", night_temp['is_ND_night'].sum())


--- 최종 ND 분류 결과 ---
주간 ND 일수: 197
야간 ND 일수: 86


### regime별 개수 확인

In [10]:
# 논문의 데이터(HW: 104일, ND: 384일)
print("--- 주간(Day) 분류 결과 ---")
print(day_temp['is_HW_day'].value_counts()) 

# 논문의 데이터(HW: 228일, ND: 260일)
print("\n--- 야간(Night) 분류 결과 ---")
print(night_temp['is_HW_night'].value_counts())

--- 주간(Day) 분류 결과 ---
is_HW_day
False    197
True      12
Name: count, dtype: int64

--- 야간(Night) 분류 결과 ---
is_HW_night
True     125
False     86
Name: count, dtype: int64


# Granger test baseline
- Bivariate Granger Causality
- 오직 센서 i 온도와 센서 j 온도
- 두 변수만 1:1로 놓고 "i가 먼저 변하면 j도 따라 변하나?"를 확인

In [11]:
import pandas as pd
from statsmodels.tsa.stattools import grangercausalitytests
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# ---------------------------------------------------------
# 1. 전역 데이터 피벗 
# ---------------------------------------------------------
print("전체 데이터 피벗 중... (중복된 시간대의 측정값은 평균으로 처리)")

# pivot() 대신 pivot_table()을 사용하고 aggfunc='mean'을 추가
pivot_df = merged_df.pivot_table(
    index='측정시간', 
    columns='시리얼', 
    values='온도 평균(℃)',
    aggfunc='mean' 
)
print("피벗 완료!\n")

# 결과를 담을 빈 리스트
granger_results = []
max_lag = 2

# ---------------------------------------------------------
# 2. 500m 이내 센서 쌍 루프 실행
# ---------------------------------------------------------
# list 형태인 pairs를 DataFrame으로 변환
pairs_df = pd.DataFrame(pairs, columns=['sensor_i', 'sensor_j'])

# pairs 대신 변환된 pairs_df 사용
for idx, row in tqdm(pairs_df.iterrows(), total=len(pairs_df), desc="Granger Test 진행중"):
    sensor_A = row['sensor_i'] 
    sensor_B = row['sensor_j'] 
    
    try:
        temp_data = pivot_df[[sensor_A, sensor_B]].dropna()
        
        if len(temp_data) < 10:
            continue
            
        # [Test 1] A -> B
        test_A_to_B = grangercausalitytests(temp_data[[sensor_B, sensor_A]], maxlag=max_lag, verbose=False)
        p_val_A_to_B = test_A_to_B[max_lag][0]['ssr_ftest'][1]
        
        # [Test 2] B -> A
        test_B_to_A = grangercausalitytests(temp_data[[sensor_A, sensor_B]], maxlag=max_lag, verbose=False)
        p_val_B_to_A = test_B_to_A[max_lag][0]['ssr_ftest'][1]
        
        is_A_source = p_val_A_to_B < 0.05
        is_B_source = p_val_B_to_A < 0.05
        
        status = "None"
        if is_A_source and not is_B_source:
            status = "A_is_Source"
        elif is_B_source and not is_A_source:
            status = "B_is_Source"
        elif is_A_source and is_B_source:
            status = "Bidirectional (Hub)"
            
        granger_results.append({
            'Sensor_A': sensor_A,
            'Sensor_B': sensor_B,
            'p_val_A_to_B': round(p_val_A_to_B, 4),
            'p_val_B_to_A': round(p_val_B_to_A, 4),
            'Status': status,
            'Valid_N': len(temp_data)
        })
        
    except KeyError:
        continue
    except Exception as e:
        continue

# ---------------------------------------------------------
# 3. 분석 결과 데이터프레임화 및 요약
# ---------------------------------------------------------
final_granger_df = pd.DataFrame(granger_results)

print("\n--- 📊 Granger F-test 최종 요약 ---")
print(final_granger_df['Status'].value_counts())

display(final_granger_df.head())

전체 데이터 피벗 중... (중복된 시간대의 측정값은 평균으로 처리)
피벗 완료!



Granger Test 진행중: 100%|██████████| 2806/2806 [00:29<00:00, 94.29it/s] 


--- 📊 Granger F-test 최종 요약 ---
Status
Bidirectional (Hub)    2346
A_is_Source             120
B_is_Source             120
None                     42
Name: count, dtype: int64


,Sensor_A,Sensor_B,p_val_A_to_B,p_val_B_to_A,Status,Valid_N
0,OC3CL200012,V02Q1940204,0.0,0.0000,Bidirectional (Hub),1878
1,OC3CL200012,V02Q1940387,0.0,0.0000,Bidirectional (Hub),1877
2,OC3CL200013,OC3CL200037,0.0,0.0000,Bidirectional (Hub),1807
3,OC3CL200013,OC3CL200038,0.0,0.0015,Bidirectional (Hub),1008
4,OC3CL200013,V02Q1940232,0.0,0.0000,Bidirectional (Hub),2169
